In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
# dense_index = DenseIndex(model, "../data/processed/_dense_court_removed_citation", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
# reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)
reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True)

In [5]:
valid_df = pd.read_csv("../data/valid_rewrite_001.csv")
valid_id_2_gold_citations_d = {}
valid_id_2_query_d = {}
for query_id, gold_citations, query in zip(valid_df['query_id'], valid_df['gold_citations'], valid_df['query2']):
    valid_id_2_gold_citations_d[query_id] = gold_citations.split(';')
    valid_id_2_query_d[query_id] = query

In [6]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm
import bge_utils
import numpy as np

valid_multi_question_df = pd.read_csv("../data/valid_rewrite_002.csv")

recall_citations_l = []
gold_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(valid_multi_question_df['query_id'], valid_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in valid_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

query_id_2_recall_hits = {}

recall_hits_l = []

cc_repeat_count = defaultdict(int)

for query_id, query_list in query_id_2_query_list.items():
    
    _hits = []
    for q in query_list:
        hits1 = dense_index.search_with_score(q, top_k=1000)
        for hit,_ in hits1:
            cc_repeat_count[hit['citation']] += 1
        hits2 = sparse_index.search_with_score(q, top_k=1000)
        for hit,_ in hits2:
            cc_repeat_count[hit['citation']] += 1
        _hits = hits_utils.merge_hits_with_score_l_by_max(_hits,hits_utils.merge_hits_with_score_l_by_max(hits1, hits2))
    
    raw_query = query_list[-1]             

    all_hits = _hits.copy()
    
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    gold_citations_l.append(valid_id_2_gold_citations_d[query_id])

    query_id_2_recall_hits[query_id] = all_hits.copy()
    
    second_layer = citation_utils.compute_citation_score_with_sentence_pos(query_id_2_recall_hits[query_id], decay="log")

    citations = []
    for citation, score in second_layer:
        if citation in court_consideration_d:
            citations.append(citation)
        if citation in law_d:
            citations.append(citation)

    recall_citations_l.append(list(set(citations)))
    
r = metric_utils.cal_recall(recall_citations_l, gold_citations_l)
print(r)

# 准备好document
# recall_hits_l = []
# for recall_citations in recall_citations_l:
#     recall_hits = []
#     for citation in set(recall_citations):
#         if citation in court_consideration_d:
#             recall_hits.append({'citation':citation, 'text':court_consideration_d[citation]})
#         elif citation in law_d:
#             recall_hits.append({'citation':citation, 'text':law_d[citation]})
#     recall_hits_l.append(recall_hits)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[val_001] hits_merge.len: 3710
[val_002] hits_merge.len: 4897
[val_003] hits_merge.len: 4580
[val_004] hits_merge.len: 4753
[val_005] hits_merge.len: 3271
[val_006] hits_merge.len: 3903
[val_007] hits_merge.len: 4849
[val_008] hits_merge.len: 6962
[val_009] hits_merge.len: 3622
[val_010] hits_merge.len: 4046
0.529168757115894


In [7]:
import numpy as np

def logsumexp(scores):
    scores = np.array(scores)
    m = np.max(scores)
    return m + np.log(np.sum(np.exp(scores - m)))

In [8]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

from openai import OpenAI
import deepseek_utils
import bge_utils
import evidence_utils
import numpy as np
import math

query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):
    # 计算evidence分数
    evidence_l = []
    recall_hits = query_id_2_recall_hits[query_id]
    for hit, score in recall_hits:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(hit['text'])
        # print(parsed_cc['citations'])
        for citation, first_appears_idx in parsed_cc['citations']:
            evidence = citation_utils.build_evidence(parsed_cc['sentences'], first_appears_idx, window_size=5)
            evidence_l.append({'citation':citation, 'text':evidence})

    print("===>",query_id, ", evidence_l.len:", len(evidence_l))

    raw_query = query_list[-1]
    
    evidence_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, evidence_l)

    evidence_with_score_l = evidence_with_score_l_raw

    # agg by top-k score sum
    citation_evidence_score_l_d = defaultdict(list)
    for evidence, score in evidence_with_score_l:
        citation_evidence_score_l_d[evidence['citation']].append((evidence, score))

    citation_score_d = {}
    for citation, l in citation_evidence_score_l_d.items():
        l2 = sorted(l, key=lambda x: x[1], reverse=True)[:5]
        score = sum([score for _, score in l2])
        citation_score_d[citation] = score
    
    # 对综合分数排序
    sorted_citation_score_l = sorted([(c,score) for c,score in citation_score_d.items()], key=lambda x: x[1], reverse=True)
    
    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_score_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    result_citation_l.append(query_result2)

    print("result_citation_l[-1].len:", len(result_citation_l[-1]))

    break
    
max_limit = None
max_r = None
max_p = None
max_f1 = 0
for limit in [5,10,15,20,25,30,35,40,45]:
    result_citation_l2 = [r[:limit] for r in result_citation_l]
    result = metric_utils.macro_f1(result_citation_l2, gold_citations_l[:len(result_citation_l2)])
    if max_f1 < result['macro_f1']:
        max_r = result['macro_recall']
        max_p = result['macro_precision']
        max_limit = limit
        max_f1 = result['macro_f1']
print(f"[{max_limit}] r:", max_r, ", p:", max_p, "f1:",max_f1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  0%|          | 0/10 [00:00<?, ?it/s]

===> val_001 , evidence_l.len: 10724


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


result_citation_l[-1].len: 769
[15] r: 0.09523809523809523 , p: 0.26666666666666666 f1: 0.14035087719298245
